# 02 -- Train SFT (Step 5: the Day-1 baseline)

Plan v3, Step 5. Fine-tune Qwen3-0.6B (LoRA adapters + value head) to imitate Stockfish on a small slice of the
Step 3 dataset -- one config: **structured text + masking on** -- then take a first look at where each saved
checkpoint lands on the Elo scale.

Settings live in `configs/sft.yaml` (`policy:` = the player, `train:` = training). Day-1 values: 5,000 training
positions, 10 epochs, batch 16, AdamW at a constant 1e-4, move loss + 1.0 x BCE value loss, a checkpoint every
500 steps (plus step 0 = the untrained model).

**Run it twice.** First with `SMOKE_TEST = True` (64 positions, a few steps, ~5 min): it checks that everything
runs, how fast a step is, and how much GPU memory batch 16 uses. Then with `SMOKE_TEST = False` for the real run.
The two runs save to different Drive folders.

**Safe to interrupt.** Every checkpoint also saves the optimizer. If Colab disconnects, run the notebook again
from the top: training continues from the last checkpoint and ends with the same weights as an unbroken run.

**Needs a GPU runtime** (Runtime -> Change runtime type -> L4, or T4 for debugging).

**What this notebook settles for the Step 5 gate:** a trained checkpoint with an Elo (and CI) that beats the
untrained model. Freezing ~3 checkpoints as reference-set rungs is the next part of Step 5, picked from the
screen at the bottom.

In [ ]:
# Setup: clone the repo, install deps, mount Drive, fetch the pinned Stockfish.
import os
REPO_URL = "https://github.com/derekhe99/Chess_Bot"
BRANCH = "step-5-sft"  # set to "main" once this branch is merged
REPO_DIR = "/content/Chess_Bot"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/chess-ai'

import torch
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- switch the runtime to GPU"
CPU_MODEL = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
print("GPU:", GPU, "| CPU:", CPU_MODEL, "| cores:", os.cpu_count())

## Log the Day-1 finding (one-time backfill)

The 5,000-position Day-1 run already finished (Sep 28). Run this once to add it to the shared
experiment log before it's just a folder name -- everything else in this notebook targets the
*new* run below.

In [ ]:
from utils.logging import log_experiment

DAY1_RUN_DIR = os.path.join(DRIVE_ROOT, "checkpoints", "sft", "structured_masked_n5000_e10_s0")
EXPERIMENTS = os.path.join(DRIVE_ROOT, "results", "experiments.csv")

if os.path.exists(os.path.join(DAY1_RUN_DIR, "summary.json")):
    row = log_experiment(DAY1_RUN_DIR, EXPERIMENTS, step=5, branch="step-5-sft",
        finding=("Real move-selection learning (val_top1 3% -> ~21%, value loss 0.69 -> 0.59) but "
                 "no checkpoint ever scored a point off Stockfish 1320 in the screen (0W 0D every "
                 "checkpoint, 10 games each) -- the reference set doesn't connect yet. val_move_loss "
                 "degraded after ~epoch 5 while val_top1 kept climbing: overfitting from repeating the "
                 "same 5,000 positions 10 times, not a ranking problem. Treated as Day-1 done; this "
                 "next run trains on the full labeled split instead."),
        beat_untrained=False, scored_vs_sf1320=False)
    print(row)
else:
    print("Day-1 run folder not found at", DAY1_RUN_DIR, "-- skip, or fix the path.")

## Parameters
Everything comes from `configs/sft.yaml` except `SMOKE_TEST`. The run folder is named after the settings that
define the run, so rerunning with the same settings resumes it instead of starting over.

In [ ]:
import dataclasses
import json
import math

import yaml

from data.dataset import load_datasets
from model.llm_policy import LLMPolicy, PolicyConfig
from train.sft import TrainConfig, list_checkpoints, run_name, screen_checkpoints, train

SMOKE_TEST = True   # True: a few steps on 64 positions (checks speed + memory). False: the real run.

cfg = yaml.safe_load(open("configs/sft.yaml"))
data_cfg = yaml.safe_load(open("configs/data.yaml"))
POLICY = PolicyConfig.from_dict(cfg["policy"])
TRAIN = TrainConfig.from_dict(cfg["train"])
FULL_TRAIN = TRAIN
if SMOKE_TEST:
    TRAIN = dataclasses.replace(TRAIN, train_positions=64, val_positions=32, val_top1_positions=8, epochs=1,
                                checkpoint_every=2, log_every=1)

RUN = run_name(POLICY, TRAIN) + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = os.path.join(DRIVE_ROOT, "checkpoints", "sft", RUN)
DATA_DIR = os.path.join(DRIVE_ROOT, "data", "processed", TRAIN.dataset)
print("player:", POLICY.model_id, "|", POLICY.representation, "| masked" if POLICY.masked else "| unmasked")
print("training:", TRAIN)
print("run folder:", RUN_DIR)

## Data
The Step 3 dataset, split by game exactly as in Step 3 (same fraction, same seed). Training uses a seeded random
slice of the train split. Stockfish's labeling CPU time counts against SFT's budget (plan Sec 0); both the whole
dataset's cost and the share for the positions this run uses are recorded -- Step 6 decides which one is charged.

In [ ]:
train_ds, val_ds = load_datasets(os.path.join(DATA_DIR, "labels.parquet"), POLICY.representation,
                                 data_cfg["val_fraction"], data_cfg["seed"])
manifest = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
n_used = min(TRAIN.train_positions or len(train_ds), len(train_ds))
sf_cpu_total = manifest["stockfish_cpu_seconds"]
sf_cpu_used = sf_cpu_total * n_used / manifest["rows"]
print(f"train split {len(train_ds)} | validation split {len(val_ds)} | this run trains on {n_used}")
print(f"Stockfish labeling CPU: whole dataset {sf_cpu_total / 60:.1f} min | share for {n_used} positions {sf_cpu_used / 60:.1f} min")

## Load the model
Qwen3-0.6B (downloaded once per session, ~1.2 GB) with fresh LoRA adapters and a zero-initialized value head.
Only those train; the base weights stay frozen.

In [ ]:
policy = LLMPolicy.from_pretrained(POLICY)
n_train = sum(p.numel() for p in policy.trainable_parameters())
n_all = sum(p.numel() for p in policy.model.parameters())
print(f"dtype {policy.model.get_input_embeddings().weight.dtype} on {policy.device}")
print(f"trainable {n_train / 1e6:.1f}M of {n_all / 1e6:.0f}M parameters ({n_train / n_all:.1%})")

## Train
Prints the train losses every few steps and, at every checkpoint, the held-out numbers: move loss, value loss
(and its mean error in win probability), and **top-1** -- how often the masked pick equals Stockfish's move.
If this cell is interrupted, rerun the notebook from the top and it picks up where it stopped.

In [ ]:
torch.cuda.reset_peak_memory_stats()
summary = train(policy, train_ds, val_ds, TRAIN, RUN_DIR, extra_info={
    "gpu": GPU, "cpu": CPU_MODEL, "stockfish": SF_VERSION, "dataset_rows": manifest["rows"],
    "stockfish_cpu_seconds_dataset": sf_cpu_total, "stockfish_cpu_seconds_share": sf_cpu_used})
print(json.dumps(summary, indent=2))

per_step = summary["gpu_seconds_train"] / summary["step"]
full_steps = FULL_TRAIN.epochs * math.ceil(n_used / FULL_TRAIN.batch_size)  # n_used: the resolved
                                                                            # count (None -> whole split)
print(f"\n{per_step:.2f} s/step at batch {TRAIN.batch_size} | peak GPU memory {summary.get('peak_gpu_memory_gb')} GB "
      f"of {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
print(f"full run: {full_steps} steps -> ~{full_steps * per_step / 60:.0f} min of training (+ validation at each checkpoint)")

## Training curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

tlog = pd.read_csv(os.path.join(RUN_DIR, "train_log.csv"))
vlog = pd.read_csv(os.path.join(RUN_DIR, "val_log.csv"))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(tlog["step"], tlog["train_move_loss"], label="train")
ax[0].plot(vlog["step"], vlog["val_move_loss"], "o-", label="validation")
ax[0].set_title("move loss: -log P(Stockfish's move)")
ax[1].plot(tlog["step"], tlog["train_value_loss"], label="train")
ax[1].plot(vlog["step"], vlog["val_value_loss"], "o-", label="validation")
ax[1].set_title(f"value loss ({TRAIN.value_loss})")
ax[2].plot(vlog["step"], vlog["val_top1"], "o-")
ax[2].set_title("top-1: masked pick == Stockfish's move")
for a in ax:
    a.set_xlabel("step")
    a.legend() if a.get_legend_handles_labels()[0] else None
plt.tight_layout()
plt.show()
print(vlog.to_string(index=False))

## Screen the checkpoints: where did training land?
A quick, rough rating of every checkpoint (step 0 = untrained included): each plays random, greedy, and
Stockfish 1320, then one Elo fit over all the games rates them (1320 pinned). Random and greedy get rough ratings
too, through the checkpoints. This is a first look, not the final numbers -- few games, wide intervals.

What to look for:
1. **Gate:** the best checkpoint's rating is above the untrained model's, with non-overlapping intervals.
2. **Rungs (next part of Step 5):** freezing checkpoint rungs needs checkpoints spread between greedy and
   Stockfish 1320, and at least one that takes points off Stockfish 1320. If every checkpoint is flagged
   OFF SCALE, none of them scored against it -- that's a finding to discuss before picking rungs.

In [ ]:
from eval.anchors import REFERENCE_SET, FrozenRegistry, build_agents, close_agents, fixed_ratings
from eval.elo import format_table
from eval.match import summarize

SCREEN_GAMES = 2 if SMOKE_TEST else 6         # games per checkpoint per opponent (color-swapped pairs)
SCREEN_MAX_PLIES = 150   # cap a screening game early (this is a rough first look, not a real eval game --
                         # see the smoke-test run: masked play on a barely-trained model ran to ~200 plies
                         # on average and some hit the 400-ply draw adjudication; at ~200ms/move that is
                         # ~2.5 hours for the real screen (8 checkpoints x 3 opponents x 10 games) at 400
registry = FrozenRegistry(os.path.join(DRIVE_ROOT, "results", "reference_set.csv"))
FIXED = fixed_ratings(REFERENCE_SET, registry.ratings())      # sf_1320 pinned (+ sf_1500/1700, not played here)
opponents = build_agents([s for s in REFERENCE_SET if s.name in ("random", "greedy", "sf_1320")])
steps = [s for s, _ in list_checkpoints(RUN_DIR)]
print("checkpoints:", steps, "| fixed:", FIXED)
try:
    estimates, screen_records = screen_checkpoints(RUN_DIR, opponents, FIXED, SCREEN_GAMES, max_plies=SCREEN_MAX_PLIES)
finally:
    close_agents(opponents)

print()
print(format_table(estimates.values()))
out = {"games_per_opponent": SCREEN_GAMES, "fixed": FIXED, "gpu": GPU, "cpu": CPU_MODEL,
       "estimates": {n: dataclasses.asdict(e) for n, e in estimates.items()},
       "records": [{k: v for k, v in dataclasses.asdict(r).items() if k != "latency"} for r in screen_records]}
with open(os.path.join(RUN_DIR, "screen.json"), "w") as f:
    json.dump(out, f)

lat = [summarize(screen_records, f"step_{s}")["avg_move_seconds"] for s in steps]
print(f"\nLLM move time: {sum(lat) / len(lat) * 1000:.0f} ms/move on average (masked: one prompt pass + legal-move scoring)")

In [ ]:
# Gate part 1: does the best trained checkpoint beat the untrained model?
untrained = estimates["step_0"]
trained = [estimates[f"step_{s}"] for s in steps if s > 0]
best = max(trained, key=lambda e: e.rating)
print(f"untrained: {untrained.rating:.0f} [{untrained.ci_low:.0f}, {untrained.ci_high:.0f}]  flag={untrained.at_bound}")
print(f"best:      {best.name} {best.rating:.0f} [{best.ci_low:.0f}, {best.ci_high:.0f}]  flag={best.at_bound}")
ok = best.at_bound is None and best.ci_low > untrained.ci_high
print("PASS" if ok else "NOT YET", "- best checkpoint's CI is entirely above the untrained model's")

## Freeze checkpoint rungs (completes the reference set)

**Update, Sep 30:** the live `rate_adaptively` run below was interrupted (`KeyboardInterrupt`)
partway through batch 2 -- nothing from an interrupted run is saved automatically (it only
returns, and gets assigned, once its `while True` loop finishes a batch with nothing left
short). The cells below reconstruct the fit instead, from the per-batch win/draw/loss counts
printed as the run went: batch 0 and batch 1 complete (all 10 matchups, 20 games each), batch 2
partial (5 of 10 matchups). `games_from_counts()` (`eval/elo.py`) turns those aggregate counts
back into individual game results -- the reconstructed **rating is exact** (the Elo fit only
ever uses a player's total score and opponent list, never color or order), the **CI is a valid
but not bit-for-bit-identical** bootstrap (see that function's docstring for why).

**Picks: `random`, `greedy`, `step_20000`.** `step_15000` (693) is dropped -- too close to
`step_20000` (719) to be worth its own rung. Its games stay IN the fit below (they still help
pin down the other three), just not frozen as a member: dropping them from the fit entirely
nearly doubles the CI half-width (~115 -> ~190 Elo), confirmed by refitting without them.

**Known gaps, accepted as provisional (Derek, Sep 30) -- revisit when there's a spare Colab
window, not blocking now:**
- Every CI here lands wider (~110-125 Elo half-width) than the plan's +/-100 target (Sec 4) --
  expected, since this used two batches instead of a full adaptive run to convergence.
- Dropping `step_15000` leaves `step_20000` (719) to `sf_1320` (1320) as one ~600-Elo gap,
  well past the plan's own "<=300 Elo between neighbors" selection heuristic -- that heuristic
  is explicitly marked tentative in the plan and left to Derek to revise; the gap is expected to
  close as later SFT/RL checkpoints get frozen in, not by this run.


In [ ]:
import hashlib
from pathlib import Path

from eval.anchors import FrozenMember

from eval.elo import fit_with_ci, format_table, games_from_counts

# (a, b, a_wins, draws, a_losses) -- a's aggregate record against b, summed across whatever
# batches finished before the interrupt (batch 0 + batch 1 complete, batch 2 partial).
TOTALS = [
    ("random", "greedy",         0, 54,  6),
    ("random", "step_15000",     0, 20, 20),
    ("random", "step_20000",     0, 19, 41),
    ("random", "sf_1320",        0,  0, 40),
    ("greedy", "step_15000",     1, 37,  2),
    ("greedy", "step_20000",     1, 38,  1),
    ("greedy", "sf_1320",        0,  2, 58),
    ("step_15000", "step_20000", 5, 27,  8),
    ("step_15000", "sf_1320",    0,  6, 54),
    ("step_20000", "sf_1320",    0,  2, 58),
]

freeze_results = []
for a, b, w, d, l in TOTALS:
    freeze_results.extend(games_from_counts(a, b, w, d, l))

# One joint fit over everyone -- step_15000 included as data (it isn't frozen below, but its
# games still constrain random/greedy/step_20000). sf_1320 is the only fixed rating in play here.
freeze_est = fit_with_ci(freeze_results, FIXED)
print(format_table(freeze_est.values()))


In [ ]:
CANDIDATES = ["random", "greedy", "step_20000"]   # step_15000 excluded -- see markdown above
step_dirs = dict(list_checkpoints(RUN_DIR))

def _weights_hash(step_dir: Path) -> str:
    h = hashlib.sha256()
    h.update((step_dir / "adapter_model.safetensors").read_bytes())
    h.update((step_dir / "value_head.pt").read_bytes())
    return h.hexdigest()[:16]

NOTES = {"step": 5, "games_per_matchup": 20,
         "batches_completed": "batch0+batch1 full (10/10 matchups), batch2 partial (5/10)",
         "source": "reconstructed via games_from_counts() after a KeyboardInterrupt during batch 2 of "
                   "rate_adaptively -- rating exact, CI a valid but not bit-identical bootstrap "
                   "(see eval/elo.py:games_from_counts)",
         "ci_target_met": False,   # 95% CI half-width ~110-125 Elo, wider than the plan's +/-100 (Sec 4)
         "gpu": GPU, "cpu": CPU_MODEL}

for name in CANDIDATES:
    if name in registry:
        print(f"{name} already frozen at {registry.ratings()[name]:.0f} -- skipped (never re-rated)")
        continue
    e = freeze_est[name]
    if name.startswith("step_"):
        step = int(name.split("_")[1])
        spec = {"kind": "sft_checkpoint", "run": RUN, "step": step,
                "weights_sha256": _weights_hash(step_dirs[step]),
                "representation": POLICY.representation, "masked": POLICY.masked}
    else:
        spec = dataclasses.asdict(next(s for s in REFERENCE_SET if s.name == name))
    registry.add(FrozenMember(name, e.rating, e.ci_low, e.ci_high, e.games, spec=spec, notes=NOTES))
registry.save()
print("Frozen reference set:", {n: round(v) for n, v in registry.ratings().items()}, "->", registry.path)


## Log this run

`beat_untrained` and `scored_vs_sf1320` are read off the screen above; `FINDING` is the one thing
to write yourself -- edit it to say what this run actually showed before running the cell.

In [ ]:
from utils.logging import log_experiment, read_experiments

sf1320_games = [r for r in screen_records if "sf_1320" in (r.white, r.black)]
SCORED_VS_SF1320 = any((r.white_score if r.black == "sf_1320" else 1.0 - r.white_score) > 0 for r in sf1320_games)

FINDING = "TODO: one or two sentences on what this run showed."   # edit before running

row = log_experiment(RUN_DIR, os.path.join(DRIVE_ROOT, "results", "experiments.csv"), step=5,
                     branch="step-5-sft", finding=FINDING, beat_untrained=ok, scored_vs_sf1320=SCORED_VS_SF1320)
print(row)
read_experiments(os.path.join(DRIVE_ROOT, "results", "experiments.csv"))